<a href="https://colab.research.google.com/github/cube2ishaank-debug/OmniDimension/blob/main/OmniDimension.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import scipy.io.wavfile as wav
from scipy.signal import butter, lfilter

def lowpass_filter(data, cutoff, fs, order=2):
    """Applies a Butterworth low-pass filter to simulate head shadowing."""
    nyquist = 0.5 * fs
    normal_cutoff = cutoff / nyquist
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    return lfilter(b, a, data)

def apply_enhanced_8d_effect(input_wav_path, output_wav_path, speed_hz=0.15, distance_factor=0.7):
    sample_rate, data = wav.read(input_wav_path)

    # Convert input data to float64 for high-precision DSP calculations
    data = data.astype(np.float64)

    # Ensure stereo structure
    if len(data.shape) == 1:
        data = np.column_stack((data, data))

    num_samples = len(data)
    time = np.linspace(0, num_samples / sample_rate, num_samples, endpoint=False)

    # 1. 360-degree rotation angle
    angle = 2 * np.pi * speed_hz * time

    # 2. Equal-Power Panning (Sine/Cosine Law)
    # Maps angle smoothly across channels maintaining constant loudness
    pan_angle = 0.25 * np.pi * (1 + np.cos(angle))
    left_gain = np.cos(pan_angle)
    right_gain = np.sin(pan_angle)

    attenuation = 1.0 - (0.25 * distance_factor)
    direct_l = data[:, 0] * left_gain * attenuation
    direct_r = data[:, 1] * right_gain * attenuation

    # 3. Interaural Time Difference (ITD) - Dynamic Micro-Delays
    max_itd_samples = int(sample_rate * 0.0007)  # ~0.7ms maximum head width delay
    itd_shift_l = np.int32((0.5 * (1 + np.sin(angle))) * max_itd_samples)
    itd_shift_r = np.int32((0.5 * (1 - np.sin(angle))) * max_itd_samples)

    # Apply dynamic delay indexing
    sample_indices = np.arange(num_samples)
    idx_l = np.clip(sample_indices - itd_shift_l, 0, num_samples - 1)
    idx_r = np.clip(sample_indices - itd_shift_r, 0, num_samples - 1)

    direct_l = direct_l[idx_l]
    direct_r = direct_r[idx_r]

    # 4. Multi-Tap Reverb with Exponential Decay
    delays = [0.012, 0.024, 0.038]  # Delays in seconds
    decays = [0.35, 0.20, 0.10]    # Decaying amplitude multipliers

    processed_left = direct_l.copy()
    processed_right = direct_r.copy()

    for d, g in zip(delays, decays):
        d_samples = int(sample_rate * d)

        # Cross-channel early reflections
        ref_l = np.pad(direct_r, (d_samples, 0))[:num_samples] * (g * distance_factor)
        ref_r = np.pad(direct_l, (d_samples, 0))[:num_samples] * (g * distance_factor)

        # 5. Low-Pass Filtering on Reflections (Distance & Occlusion)
        ref_l = lowpass_filter(ref_l, cutoff=4000, fs=sample_rate)
        ref_r = lowpass_filter(ref_r, cutoff=4000, fs=sample_rate)

        processed_left += ref_l
        processed_right += ref_r

    output_data = np.column_stack((processed_left, processed_right))

    # Scale back down if necessary to avoid 16-bit integer overflow clipping
    max_val = np.max(np.abs(output_data))
    if max_val > 32767.0:
        output_data = (output_data / max_val) * 32767.0

    # Write output matching original save methodology
    wav.write(output_wav_path, sample_rate, output_data.astype(np.int16))

# Execute processing
apply_enhanced_8d_effect("/content/input.wav", "output.wav", speed_hz=0.250, distance_factor=1)